# OLAP · 03 · BQ3 · Which states are hit hardest relative to their size?
**Owner: Ar** · Reviewer: M · Decision supported: **where to review pricing and exposure** (Head of Underwriting)

| Query | Question | OLAP operation |
|---|---|---|
| 3.1 | Events and loss per resident, by state | **Drill-across** two facts at the shared grain (state × year) |
| 3.2 | The same, for floods only | **Slice** (one value of one dimension) |
| 3.3 | How much loss sits in east-coast states in the warm half of the year? | **Dice** (a sub-cube on three dimensions) |
| 3.4 | The dice, broken out cell by cell | **Dice** + roll-up |

**Drill-across:** `fact_event_loss` (one row per event) and `fact_state_year` (one row per state per year) have different grains. We aggregate the event fact up to state × year first, then join the two on the conformed `dim_location` and the year. Joining event rows directly to state-year rows would repeat population values and inflate them.

In [ ]:
%run ./00_olap_setup

### 3.1 · Drill-across: events and loss per resident, by state
All eight states and territories are listed (`LEFT JOIN`), so a state with no recorded events shows zero rather than disappearing. Events with an Unknown state cannot be placed and are reported on their own line.

In [ ]:
q31 = olap("bq3_state_per_resident", f"""
WITH ev AS (                       -- event fact rolled up to the snapshot grain
  SELECT state_code, calendar_year AS year, count(*) AS events, coalesce(sum(loss_m), 0) AS loss_m
  FROM v_event_loss GROUP BY state_code, calendar_year),
sy AS (                            -- snapshot fact at its own grain
  SELECT state_code, state_name, year, population FROM v_state_year),
joined AS (
  SELECT sy.state_code, sy.state_name, sy.year, sy.population,
         coalesce(ev.events, 0) AS events, coalesce(ev.loss_m, 0) AS loss_m
  FROM sy LEFT JOIN ev ON ev.state_code = sy.state_code AND ev.year = sy.year)
SELECT state_code, state_name, sum(events) AS events, round(sum(loss_m), 1) AS loss_m,
       round(avg(population) / 1e6, 2) AS avg_population_m,
       round(sum(events) / (avg(population) / 1e6), 1) AS events_per_million_residents,
       round(1e6 * sum(loss_m) / avg(population), 0) AS loss_per_resident_aud,
       round(1e6 * sum(loss_m) / avg(population) / count(DISTINCT year), 0) AS annual_loss_per_resident_aud,
       round(100 * sum(loss_m) / (SELECT sum(loss_m) FROM v_event_loss), 1) AS pct_of_all_loss
FROM joined GROUP BY state_code, state_name
UNION ALL
SELECT 'UNK', 'Unknown state', count(*), round(sum(loss_m), 1), NULL, NULL, NULL, NULL,
       round(100 * sum(loss_m) / (SELECT sum(loss_m) FROM v_event_loss), 1)
FROM v_event_loss WHERE state_code = 'UNK'
ORDER BY loss_per_resident_aud DESC NULLS LAST
""")

### 3.2 · Slice: floods only
A slice fixes one dimension to one value (`hazard_group = 'Flood'`) and looks at the rest of the cube.

In [ ]:
q32 = olap("bq3_slice_flood", """
WITH pop AS (SELECT state_code, avg(population) AS population FROM v_state_year GROUP BY state_code)
SELECT f.state_code, count(*) AS flood_events, round(sum(f.loss_m), 1) AS flood_loss_m,
       round(1e6 * sum(f.loss_m) / max(pop.population), 0) AS flood_loss_per_resident_aud,
       round(100 * sum(f.loss_m) / (SELECT sum(loss_m) FROM v_event_loss WHERE hazard_group = 'Flood'), 1) AS pct_of_flood_loss
FROM v_event_loss f LEFT JOIN pop ON pop.state_code = f.state_code
WHERE f.hazard_group = 'Flood'
GROUP BY f.state_code ORDER BY flood_loss_m DESC NULLS LAST
""")

### 3.3 · Dice: east coast × warm half of the year × weather hazards
A dice restricts **several** dimensions at once. Here: state ∈ {QLD, NSW}, season ∈ {Summer, Spring}, hazard family = Meteorological or Hydrological. The query compares that sub-cube with everything else.

In [ ]:
q33 = olap("bq3_dice_east_coast_warm", """
SELECT CASE WHEN state_code IN ('QLD', 'NSW') AND season IN ('Summer', 'Spring')
                 AND hazard_family IN ('Meteorological', 'Hydrological') THEN 'QLD+NSW · Summer+Spring · weather'
            ELSE 'everything else' END AS sub_cube,
       count(*) AS events, round(sum(loss_m), 1) AS loss_m,
       round(100 * count(*) / (SELECT count(*) FROM v_event_loss), 1) AS pct_of_events,
       round(100 * sum(loss_m) / (SELECT sum(loss_m) FROM v_event_loss), 1) AS pct_of_loss
FROM v_event_loss GROUP BY 1 ORDER BY loss_m DESC
""")

### 3.4 · The dice, cell by cell

In [ ]:
q34 = olap("bq3_dice_cells", """
SELECT state_code, season, hazard_group, count(*) AS events, round(sum(loss_m), 1) AS loss_m
FROM v_event_loss
WHERE state_code IN ('QLD', 'NSW') AND season IN ('Summer', 'Spring') AND hazard_family IN ('Meteorological', 'Hydrological')
GROUP BY state_code, season, hazard_group
ORDER BY loss_m DESC NULLS LAST
""")